# Modelo predictivo y exploratorio para análisis de estructuras reticulares parametrizadas

**Iteración 4 — Sección 1 + Sección 2A + Sección 2B**

Esta iteración conserva la estructura optimizada de la iteración 3 y agrega la subsección **2B — Clasificación de viabilidad geométrica**.

Cambios principales frente a la iteración 3:

- Se mantiene la **Sección 1** enfocada en preparación, sin gráficos redundantes.
- Se mantiene la **Sección 2A** como bloque principal de EDA y visualización.
- Se agrega la **Sección 2B** para predecir `failure_bin`, donde `1` representa falla geométrica y `0` representa diseño viable.
- Se modelan principalmente EXP, Kelvin y Schwarz, porque BCC tiene muy pocas fallas para un clasificador individual robusto.
- Se comparan modelos base contra modelos no lineales usando métricas adecuadas para clases desbalanceadas.
- Se incluye evaluación con matriz de confusión, F1, recall de fallas, PR-AUC, threshold tuning e importancia de variables.


# 1. Problema y datos

El proyecto analiza estructuras reticulares o *lattice structures* parametrizadas generadas en SolidWorks. Cada fila representa una combinación CAD. Algunas reconstruyen correctamente y otras fallan. Por tanto, se plantean dos líneas principales de modelado:

1. **Clasificación de viabilidad geométrica:** predecir `rebuild_ok_bin`.
2. **Regresión de propiedades físicas:** predecir propiedades como `volume_m3`, `surface_area_m2` y `envelope_volume_m3` solo para geometrías viables.

Decisiones metodológicas:

- Las fallas no se eliminan, porque son necesarias para clasificación.
- Las salidas físicas faltantes en fallas son esperadas.
- Solo se eliminan duplicados estrictos: mismas entradas CAD y mismas salidas.
- Se crean dataframes generales por familia: `df_bcc`, `df_exp`, `df_kelvin`, `df_schwarz`.
- Las masas se conservan para análisis industrial, pero no se usarán como features principales porque dependen de `volume_m3`.

In [ ]:
# ============================================================
# 1.1 Configuración inicial
# ============================================================

!pip install -q openpyxl

import os
import io
import hashlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.float_format", "{:,.6f}".format)

RANDOM_STATE = 42
RUN_PAIRPLOTS = False  # Cambiar a True solo si quieres gráficos diagnósticos pesados

os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

In [ ]:
# ============================================================
# 1.2 Configuración del repositorio GitHub
# ============================================================

GITHUB_USER = "diego-rivera-eng"
REPO_NAME = "lattice_strucures_cad-ml"
BRANCH = "main"
BASE_RAW_URL = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/data/raw"

DATA_FILES = {
    "BCC": ["bcc_batch_step_resultsA.xlsx", "bcc_batch_step_resultsB.xlsx"],
    "EXP": ["exp_batch_step_results.xlsx"],
    "KELVIN": ["kelvin_batch_step_results.xlsx"],
    "SCHWARZ": ["schwarz_batch_step_results.xlsx"]
}

print(BASE_RAW_URL)

In [ ]:
# ============================================================
# 1.3 Carga reproducible desde GitHub
# ============================================================

def read_excel_from_github(url):
    response = requests.get(url)
    if response.status_code != 200:
        raise ValueError(f"No se pudo leer el archivo. URL={url}, status={response.status_code}")
    return pd.read_excel(io.BytesIO(response.content))


def load_lattice_datasets(base_url, data_files):
    frames = []
    for geometry, files in data_files.items():
        for file_name in files:
            url = f"{base_url}/{file_name}"
            print(f"Cargando {geometry}: {file_name}")
            df_temp = read_excel_from_github(url)
            df_temp["geometry"] = geometry
            df_temp["source_file"] = file_name
            frames.append(df_temp)
    return pd.concat(frames, ignore_index=True)


df_all_raw = load_lattice_datasets(BASE_RAW_URL, DATA_FILES)
print(f"Filas: {df_all_raw.shape[0]:,}")
print(f"Columnas: {df_all_raw.shape[1]:,}")
display(df_all_raw.head())

In [ ]:
# ============================================================
# 1.4 Variables del proyecto
# ============================================================

INPUT_COLS_BY_GEOMETRY_BASE = {
    "BCC": ["D_toro", "d_wire", "n_arms", "n_radial"],
    "EXP": ["t", "square", "excentricity", "height"],
    "KELVIN": ["cell_size", "alpha", "wire_diameter", "beta", "theta", "n_circular"],
    "SCHWARZ": ["cell_size", "node_diameter", "thick"]
}

# Alias para compatibilidad
INPUT_COLS_BY_GEOMETRY = INPUT_COLS_BY_GEOMETRY_BASE.copy()

PHYSICAL_OUTPUTS = [
    "envelope_volume_m3", "volume_m3", "surface_area_m2",
    "plastic_mass_kg", "steel_mass_kg", "elapsed_s"
]

MAIN_REGRESSION_TARGETS = ["volume_m3", "surface_area_m2", "envelope_volume_m3", "elapsed_s"]
BINARY_OUTPUT = "rebuild_ok_bin"

DERIVED_PERFORMANCE_COLS = [
    "relative_density", "porosity", "surface_to_volume_ratio",
    "surface_to_envelope_volume_ratio", "surface_to_plastic_mass",
    "surface_to_steel_mass", "plastic_mass_per_envelope_volume",
    "steel_mass_per_envelope_volume"
]

CAD_DERIVED_INPUTS_BY_GEOMETRY = {
    "BCC": ["d_wire_to_D_toro"],
    "EXP": ["height_to_square"],
    "KELVIN": ["wire_diameter_to_cell_size"],
    "SCHWARZ": ["thick_to_cell_size", "node_diameter_to_cell_size"]
}

for geometry, cols in INPUT_COLS_BY_GEOMETRY_BASE.items():
    print(f"{geometry}: {cols}")

In [ ]:
# ============================================================
# 1.5 Normalización de rebuild_ok y conversiones numéricas
# ============================================================

df_all = df_all_raw.copy()

df_all["status_norm"] = (
    df_all["status"].astype(str).str.strip().str.upper()
    if "status" in df_all.columns else np.nan
)


def normalize_rebuild_ok(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, bool):
        return int(value)
    value_str = str(value).strip().lower()
    true_values = ["true", "1", "yes", "y", "si", "sí", "ok", "success"]
    false_values = ["false", "0", "no", "n", "fail", "failed", "error"]
    if value_str in true_values:
        return 1
    if value_str in false_values:
        return 0
    return np.nan


df_all[BINARY_OUTPUT] = df_all["rebuild_ok"].apply(normalize_rebuild_ok)

mask_unknown = df_all[BINARY_OUTPUT].isna()
df_all.loc[mask_unknown & df_all["status_norm"].isin(["OK", "SUCCESS"]), BINARY_OUTPUT] = 1
df_all.loc[mask_unknown & df_all["status_norm"].str.contains("FAIL|FAILED|ERROR|ERR", na=False), BINARY_OUTPUT] = 0

df_all["failure_bin"] = np.where(df_all[BINARY_OUTPUT].notna(), 1 - df_all[BINARY_OUTPUT].astype(float), np.nan)

all_input_cols_base = sorted(set(sum(INPUT_COLS_BY_GEOMETRY_BASE.values(), [])))
numeric_candidate_cols = all_input_cols_base + PHYSICAL_OUTPUTS + ["solidworks_status", "open_errors", "open_warnings"]

for col in numeric_candidate_cols:
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

print("Distribución global de rebuild_ok_bin:")
display(df_all[BINARY_OUTPUT].value_counts(dropna=False).rename("count").to_frame())

## 1.6 Identificador de diseño paramétrico

El `design_group_id` identifica combinaciones únicas de parámetros CAD dentro de cada familia. No identifica una fila del Excel, sino un diseño paramétrico. Será útil en modelado para evitar que el mismo diseño aparezca a la vez en entrenamiento y prueba.

In [ ]:
# ============================================================
# 1.6 Identificador de diseño paramétrico
# ============================================================

def make_design_group_id(row, input_cols_by_geometry):
    geometry = row["geometry"]
    parts = [f"geometry={geometry}"]
    for col in input_cols_by_geometry[geometry]:
        parts.append(f"{col}={row[col] if col in row.index else np.nan}")
    digest = hashlib.md5("|".join(parts).encode("utf-8")).hexdigest()[:12]
    return f"{geometry}_{digest}"


df_all["design_group_id"] = df_all.apply(make_design_group_id, axis=1, input_cols_by_geometry=INPUT_COLS_BY_GEOMETRY_BASE)

design_group_summary = (
    df_all.groupby("geometry")
    .agg(rows=("geometry", "size"), unique_design_groups=("design_group_id", "nunique"))
    .reset_index()
)

display(design_group_summary)

## 1.7 Deduplicación estricta

Solo se eliminan registros con la misma familia, las mismas variables de entrada CAD y las mismas variables de salida. Si dos corridas tienen las mismas entradas pero salidas ligeramente diferentes, se conservan y se reportan.

In [ ]:
# ============================================================
# 1.7 Eliminación de duplicados estrictos por familia
# ============================================================

OUTPUT_COLS_FOR_DUPLICATES = [BINARY_OUTPUT] + [c for c in PHYSICAL_OUTPUTS if c in df_all.columns]


def remove_strict_duplicates(df, geometry, input_cols, output_cols):
    df_family = df[df["geometry"] == geometry].copy()
    duplicate_key = input_cols + [c for c in output_cols if c in df_family.columns]

    n_before = len(df_family)
    df_family_clean = df_family.drop_duplicates(subset=duplicate_key, keep="first").copy()
    n_after = len(df_family_clean)

    repeated_inputs = (
        df_family_clean.groupby(input_cols, dropna=False)
        .size().reset_index(name="n_records_same_inputs")
        .query("n_records_same_inputs > 1")
        .sort_values("n_records_same_inputs", ascending=False)
    )

    report = {
        "geometry": geometry,
        "rows_before": n_before,
        "strict_duplicates_removed": n_before - n_after,
        "rows_after": n_after,
        "same_inputs_different_outputs_cases": len(repeated_inputs)
    }
    return df_family_clean, report, repeated_inputs


family_dfs = {}
duplicate_reports = []
same_input_reports = {}

for geometry, input_cols in INPUT_COLS_BY_GEOMETRY_BASE.items():
    df_family_clean, report, repeated_inputs = remove_strict_duplicates(df_all, geometry, input_cols, OUTPUT_COLS_FOR_DUPLICATES)
    family_dfs[geometry] = df_family_clean
    duplicate_reports.append(report)
    same_input_reports[geometry] = repeated_inputs

duplicate_report_df = pd.DataFrame(duplicate_reports)
display(duplicate_report_df)

In [ ]:
# ============================================================
# 1.8 Feature engineering físico y CAD adimensional
# ============================================================

def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    valid = numerator.notna() & denominator.notna() & np.isfinite(numerator) & np.isfinite(denominator) & (denominator > 0)
    result = pd.Series(np.nan, index=numerator.index, dtype=float)
    result.loc[valid] = numerator.loc[valid] / denominator.loc[valid]
    return result


def add_physical_performance_features(df):
    df = df.copy()
    required = ["volume_m3", "envelope_volume_m3", "surface_area_m2", "plastic_mass_kg", "steel_mass_kg"]
    if not set(required).issubset(df.columns):
        return df
    df["relative_density"] = safe_divide(df["volume_m3"], df["envelope_volume_m3"])
    df["porosity"] = 1 - df["relative_density"]
    df["surface_to_volume_ratio"] = safe_divide(df["surface_area_m2"], df["volume_m3"])
    df["surface_to_envelope_volume_ratio"] = safe_divide(df["surface_area_m2"], df["envelope_volume_m3"])
    df["surface_to_plastic_mass"] = safe_divide(df["surface_area_m2"], df["plastic_mass_kg"])
    df["surface_to_steel_mass"] = safe_divide(df["surface_area_m2"], df["steel_mass_kg"])
    df["plastic_mass_per_envelope_volume"] = safe_divide(df["plastic_mass_kg"], df["envelope_volume_m3"])
    df["steel_mass_per_envelope_volume"] = safe_divide(df["steel_mass_kg"], df["envelope_volume_m3"])
    df["relative_density_is_physical"] = df["relative_density"].between(0, 1, inclusive="both")
    return df


def add_cad_ratio_features(df):
    df = df.copy()
    for cols in CAD_DERIVED_INPUTS_BY_GEOMETRY.values():
        for col in cols:
            if col not in df.columns:
                df[col] = np.nan

    mask = df["geometry"].eq("BCC")
    if {"d_wire", "D_toro"}.issubset(df.columns):
        df.loc[mask, "d_wire_to_D_toro"] = safe_divide(df.loc[mask, "d_wire"], df.loc[mask, "D_toro"])

    mask = df["geometry"].eq("EXP")
    if {"height", "square"}.issubset(df.columns):
        df.loc[mask, "height_to_square"] = safe_divide(df.loc[mask, "height"], df.loc[mask, "square"])

    mask = df["geometry"].eq("KELVIN")
    if {"wire_diameter", "cell_size"}.issubset(df.columns):
        df.loc[mask, "wire_diameter_to_cell_size"] = safe_divide(df.loc[mask, "wire_diameter"], df.loc[mask, "cell_size"])

    mask = df["geometry"].eq("SCHWARZ")
    if {"thick", "cell_size"}.issubset(df.columns):
        df.loc[mask, "thick_to_cell_size"] = safe_divide(df.loc[mask, "thick"], df.loc[mask, "cell_size"])
    if {"node_diameter", "cell_size"}.issubset(df.columns):
        df.loc[mask, "node_diameter_to_cell_size"] = safe_divide(df.loc[mask, "node_diameter"], df.loc[mask, "cell_size"])
    return df


for geometry in family_dfs:
    family_dfs[geometry] = add_physical_performance_features(family_dfs[geometry])
    family_dfs[geometry] = add_cad_ratio_features(family_dfs[geometry])

df_bcc = family_dfs["BCC"].copy()
df_exp = family_dfs["EXP"].copy()
df_kelvin = family_dfs["KELVIN"].copy()
df_schwarz = family_dfs["SCHWARZ"].copy()

df_all_processed = pd.concat([df_bcc, df_exp, df_kelvin, df_schwarz], ignore_index=True)

INPUT_COLS_BY_GEOMETRY_MODELING = {
    g: INPUT_COLS_BY_GEOMETRY_BASE[g] + CAD_DERIVED_INPUTS_BY_GEOMETRY.get(g, [])
    for g in INPUT_COLS_BY_GEOMETRY_BASE
}

print("Variables de entrada ampliadas para modelado:")
for g, cols in INPUT_COLS_BY_GEOMETRY_MODELING.items():
    print(f"{g}: {cols}")

In [ ]:
# ============================================================
# 1.9 Resumen de calidad y vistas temporales
# ============================================================

def summarize_dataset_quality(df_all, target_col):
    summary = (
        df_all.groupby("geometry")
        .agg(
            total_rows=("geometry", "size"),
            unique_design_groups=("design_group_id", "nunique"),
            viable_rows=(target_col, lambda x: (x == 1).sum()),
            failed_rows=(target_col, lambda x: (x == 0).sum()),
            unknown_rows=(target_col, lambda x: x.isna().sum()),
            complete_volume_rows=("volume_m3", lambda x: x.notna().sum()),
            complete_area_rows=("surface_area_m2", lambda x: x.notna().sum()),
            complete_envelope_rows=("envelope_volume_m3", lambda x: x.notna().sum()),
            complete_elapsed_rows=("elapsed_s", lambda x: x.notna().sum())
        )
        .reset_index()
    )
    summary["viable_pct"] = 100 * summary["viable_rows"] / summary["total_rows"]
    summary["failed_pct"] = 100 * summary["failed_rows"] / summary["total_rows"]
    return summary


dataset_quality_summary = summarize_dataset_quality(df_all_processed, BINARY_OUTPUT)
display(dataset_quality_summary)
print(f"Filas totales procesadas: {df_all_processed.shape[0]:,}")
print(f"Columnas totales procesadas: {df_all_processed.shape[1]:,}")

In [ ]:
# ============================================================
# 1.10 Funciones de vistas temporales para modelado futuro
# ============================================================

def get_input_cols_for_geometry(geometry, feature_set="base"):
    if feature_set == "base":
        return INPUT_COLS_BY_GEOMETRY_BASE[geometry]
    if feature_set == "modeling":
        return INPUT_COLS_BY_GEOMETRY_MODELING[geometry]
    raise ValueError("feature_set debe ser 'base' o 'modeling'.")


def get_viability_modeling_view(df_family, geometry, feature_set="base"):
    input_cols = get_input_cols_for_geometry(geometry, feature_set)
    df_view = df_family.dropna(subset=input_cols + [BINARY_OUTPUT]).copy()
    return df_view, df_view[input_cols].copy(), df_view[BINARY_OUTPUT].astype(int)


def get_regression_modeling_view(df_family, geometry, target_col, feature_set="base"):
    input_cols = get_input_cols_for_geometry(geometry, feature_set)
    df_view = df_family[(df_family[BINARY_OUTPUT] == 1) & df_family[target_col].notna()].dropna(subset=input_cols).copy()
    return df_view, df_view[input_cols].copy(), df_view[target_col].copy()


rows = []
for geometry, df_family in family_dfs.items():
    df_cls_base, X_cls_base, y_cls_base = get_viability_modeling_view(df_family, geometry, "base")
    df_cls_model, X_cls_model, y_cls_model = get_viability_modeling_view(df_family, geometry, "modeling")
    row = {
        "geometry": geometry,
        "classification_rows": len(df_cls_base),
        "input_features_base": X_cls_base.shape[1],
        "input_features_modeling": X_cls_model.shape[1],
        "viable_class_1": int((y_cls_base == 1).sum()),
        "failed_class_0": int((y_cls_base == 0).sum()),
        "failed_pct": 100 * int((y_cls_base == 0).sum()) / len(y_cls_base),
        "unique_design_groups_cls": df_cls_base["design_group_id"].nunique()
    }
    for target in MAIN_REGRESSION_TARGETS:
        if target in df_family.columns:
            df_reg, _, _ = get_regression_modeling_view(df_family, geometry, target, "base")
            row[f"regression_rows_{target}"] = len(df_reg)
            row[f"unique_design_groups_reg_{target}"] = df_reg["design_group_id"].nunique()
    rows.append(row)

modeling_view_report_df = pd.DataFrame(rows)

def classification_recommendation(row):
    if row["failed_class_0"] < 20:
        return "No recomendado como modelo individual: muy pocas fallas"
    if row["failed_pct"] < 15:
        return "Viable: usar métricas para desbalance"
    return "Prioritario: buena proporción de fallas"

modeling_view_report_df["classification_recommendation"] = modeling_view_report_df.apply(classification_recommendation, axis=1)
display(modeling_view_report_df)

In [ ]:
# ============================================================
# 1.11 Exportación base
# ============================================================

processed_excel_path = "data/processed/lattice_processed_by_family_iteracion3.xlsx"
base_reports_excel_path = "outputs/tables/section1_base_reports_iteracion3.xlsx"

with pd.ExcelWriter(processed_excel_path, engine="openpyxl") as writer:
    df_all_processed.to_excel(writer, sheet_name="ALL", index=False)
    df_bcc.to_excel(writer, sheet_name="BCC", index=False)
    df_exp.to_excel(writer, sheet_name="EXP", index=False)
    df_kelvin.to_excel(writer, sheet_name="KELVIN", index=False)
    df_schwarz.to_excel(writer, sheet_name="SCHWARZ", index=False)

with pd.ExcelWriter(base_reports_excel_path, engine="openpyxl") as writer:
    dataset_quality_summary.to_excel(writer, sheet_name="DATASET_QUALITY", index=False)
    design_group_summary.to_excel(writer, sheet_name="DESIGN_GROUPS", index=False)
    duplicate_report_df.to_excel(writer, sheet_name="DUPLICATES", index=False)
    modeling_view_report_df.to_excel(writer, sheet_name="MODELING_VIEWS", index=False)

print("Exportación base completada:")
print(processed_excel_path)
print(base_reports_excel_path)

# 2A. EDA y visualización

Esta sección concentra todos los gráficos exploratorios. El objetivo es entender el comportamiento geométrico y físico de cada familia antes de modelar.

La narrativa del EDA es:

1. Viabilidad geométrica por familia.
2. Correlaciones intrafamilia.
3. Espacio CAD y fallas.
4. Interacciones CAD-física.
5. Benchmarking interfamilia.
6. Pareto y diseños destacados.
7. Conclusiones para 2B y 2C.

In [ ]:
# ============================================================
# 2A.1 Vistas temporales para EDA
# ============================================================

df_viable = df_all_processed[df_all_processed[BINARY_OUTPUT] == 1].copy()
df_failed = df_all_processed[df_all_processed[BINARY_OUTPUT] == 0].copy()

print("Filas totales:", df_all_processed.shape[0])
print("Filas viables:", df_viable.shape[0])
print("Filas fallidas:", df_failed.shape[0])

In [ ]:
# ============================================================
# 2A.2 Viabilidad geométrica: conteos y porcentajes
# ============================================================

plt.figure(figsize=(9, 5))
if sns is not None:
    sns.countplot(data=df_all_processed, x="geometry", hue=BINARY_OUTPUT, order=["BCC", "EXP", "KELVIN", "SCHWARZ"], hue_order=[0, 1])
else:
    df_all_processed.groupby(["geometry", BINARY_OUTPUT]).size().unstack().loc[["BCC", "EXP", "KELVIN", "SCHWARZ"]].plot(kind="bar")
plt.title("Viabilidad geométrica por familia lattice")
plt.xlabel("Familia geométrica")
plt.ylabel("Número de corridas")
plt.legend(title="Viabilidad", labels=["Falla / no viable", "Viable"])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/figures/2A_viabilidad_conteos_por_familia.png", dpi=300)
plt.show()

viability_pct_plot = (
    df_all_processed.groupby("geometry")[BINARY_OUTPUT]
    .value_counts(normalize=True).mul(100).rename("percentage").reset_index()
)
viability_pct_plot["status_label"] = viability_pct_plot[BINARY_OUTPUT].map({0: "Falla / no viable", 1: "Viable"})

plt.figure(figsize=(9, 5))
if sns is not None:
    sns.barplot(data=viability_pct_plot, x="geometry", y="percentage", hue="status_label", order=["BCC", "EXP", "KELVIN", "SCHWARZ"], hue_order=["Falla / no viable", "Viable"])
else:
    viability_pct_plot.pivot(index="geometry", columns="status_label", values="percentage").loc[["BCC", "EXP", "KELVIN", "SCHWARZ"]].plot(kind="bar")
plt.title("Porcentaje de viabilidad geométrica por familia lattice")
plt.xlabel("Familia geométrica")
plt.ylabel("Porcentaje de corridas (%)")
plt.legend(title="Viabilidad")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/figures/2A_viabilidad_porcentaje_por_familia.png", dpi=300)
plt.show()

display(viability_pct_plot)

### Interpretación de viabilidad

BCC es la familia más estable, con muy pocas fallas. EXP y Kelvin presentan niveles intermedios, mientras que Schwarz concentra la mayor proporción de reconstrucciones no viables. Por tanto, la clasificación de viabilidad será más informativa en Schwarz, Kelvin y EXP. BCC deberá tratarse con cautela por tener muy pocos ejemplos fallidos.

In [ ]:
# ============================================================
# 2A.3 Correlaciones intrafamilia y top correlaciones
# ============================================================

EDA_OUTPUT_COLS_MODELING = [
    "volume_m3", "surface_area_m2", "envelope_volume_m3", "elapsed_s",
    "relative_density", "surface_to_volume_ratio", "surface_to_envelope_volume_ratio"
]


def plot_family_correlation_heatmap(df_family, geometry):
    cols = INPUT_COLS_BY_GEOMETRY_BASE[geometry] + CAD_DERIVED_INPUTS_BY_GEOMETRY.get(geometry, [])
    cols += [c for c in EDA_OUTPUT_COLS_MODELING if c in df_family.columns]
    cols = [c for c in cols if c in df_family.columns]
    df_corr = df_family[df_family[BINARY_OUTPUT] == 1][cols].apply(pd.to_numeric, errors="coerce")
    corr = df_corr.corr()

    plt.figure(figsize=(12, 9))
    if sns is not None:
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
    else:
        plt.imshow(corr, cmap="coolwarm", aspect="auto")
        plt.colorbar()
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.index)), corr.index)
    plt.title(f"Matriz de correlación técnica - {geometry}")
    plt.tight_layout()
    plt.savefig(f"outputs/figures/2A_corr_heatmap_{geometry}.png", dpi=300)
    plt.show()
    return corr


correlation_matrices = {}
for geometry, df_family in family_dfs.items():
    correlation_matrices[geometry] = plot_family_correlation_heatmap(df_family, geometry)

rows = []
for geometry, corr in correlation_matrices.items():
    input_cols = INPUT_COLS_BY_GEOMETRY_BASE[geometry] + CAD_DERIVED_INPUTS_BY_GEOMETRY.get(geometry, [])
    for input_col in input_cols:
        for output_col in EDA_OUTPUT_COLS_MODELING:
            if input_col in corr.index and output_col in corr.columns:
                value = corr.loc[input_col, output_col]
                rows.append({"geometry": geometry, "input_variable": input_col, "output_variable": output_col, "correlation": value, "abs_correlation": abs(value)})

top_correlations_df = pd.DataFrame(rows).sort_values(["geometry", "abs_correlation"], ascending=[True, False])
display(top_correlations_df.groupby("geometry").head(10).reset_index(drop=True))
top_correlations_df.to_excel("outputs/tables/2A_top_input_output_correlations.xlsx", index=False)

### Interpretación de correlaciones

Los heatmaps permiten ver qué parámetros CAD se asocian más con las salidas físicas. En esta iteración se excluye `porosity` del heatmap principal porque es redundante con `relative_density`. Las variables derivadas se usan para EDA y selección, pero no deben usarse como features si contienen información de la variable objetivo.

In [ ]:
# ============================================================
# 2A.4 Espacio CAD y viabilidad: gráficos focalizados
# ============================================================

VIABILITY_SCATTER_CONFIG = {
    "EXP": [("square", "height"), ("square", "excentricity"), ("t", "height")],
    "KELVIN": [("cell_size", "wire_diameter"), ("alpha", "theta"), ("beta", "n_circular")],
    "SCHWARZ": [("cell_size", "thick"), ("cell_size", "node_diameter"), ("node_diameter", "thick")]
}


def plot_viability_scatter_pairs(df_family, geometry, pairs):
    for x_col, y_col in pairs:
        if x_col not in df_family.columns or y_col not in df_family.columns:
            continue
        df_plot = df_family.dropna(subset=[x_col, y_col, BINARY_OUTPUT]).copy()
        plt.figure(figsize=(7, 5))
        if sns is not None:
            sns.scatterplot(data=df_plot, x=x_col, y=y_col, hue=BINARY_OUTPUT, hue_order=[0, 1], alpha=0.8, s=55)
        else:
            for cls in [0, 1]:
                subset = df_plot[df_plot[BINARY_OUTPUT] == cls]
                plt.scatter(subset[x_col], subset[y_col], alpha=0.8, label=str(cls))
            plt.legend(title=BINARY_OUTPUT)
        plt.title(f"{geometry} - Viabilidad: {x_col} vs {y_col}")
        plt.xlabel(x_col)
        plt.ylabel(y_col)
        plt.grid(True, alpha=0.3)
        plt.legend(title="rebuild_ok_bin")
        plt.tight_layout()
        plt.savefig(f"outputs/figures/2A_viability_scatter_{geometry}_{x_col}_{y_col}.png", dpi=300)
        plt.show()

for geometry, pairs in VIABILITY_SCATTER_CONFIG.items():
    plot_viability_scatter_pairs(family_dfs[geometry], geometry, pairs)

In [ ]:
# ============================================================
# 2A.5 Interacciones físicas CAD-salida en geometrías viables
# ============================================================

INTERACTION_CONFIG = {
    "BCC": {"x": "d_wire", "y": "surface_area_m2", "color": "n_arms"},
    "EXP": {"x": "height", "y": "surface_to_volume_ratio", "color": "excentricity"},
    "KELVIN": {"x": "wire_diameter", "y": "relative_density", "color": "cell_size"},
    "SCHWARZ": {"x": "thick", "y": "surface_to_volume_ratio", "color": "cell_size"}
}


def plot_interaction_scatter(df_family, geometry, config):
    x, y, color = config["x"], config["y"], config["color"]
    if not set([x, y, color]).issubset(df_family.columns):
        return
    df_plot = df_family[df_family[BINARY_OUTPUT] == 1].dropna(subset=[x, y, color]).copy()
    plt.figure(figsize=(9, 6))
    if sns is not None:
        sns.scatterplot(data=df_plot, x=x, y=y, hue=color, palette="rocket_r", alpha=0.75, s=55)
    else:
        plt.scatter(df_plot[x], df_plot[y], c=df_plot[color], alpha=0.75)
        plt.colorbar(label=color)
    plt.title(f"{geometry} - Interacción física: {x} vs {y}, coloreado por {color}")
    plt.xlabel(x)
    plt.ylabel(y)
    plt.grid(True, alpha=0.3)
    if y in ["surface_to_volume_ratio", "surface_area_m2"] and (df_plot[y].dropna() > 0).all():
        plt.yscale("log")
    plt.tight_layout()
    plt.savefig(f"outputs/figures/2A_interaction_{geometry}_{x}_{y}_{color}.png", dpi=300)
    plt.show()

for geometry, config in INTERACTION_CONFIG.items():
    plot_interaction_scatter(family_dfs[geometry], geometry, config)

### Lectura de interacciones

Las interacciones físicas muestran que las propiedades dependen de proporciones geométricas, no solo de dimensiones absolutas. Esto justifica los ratios CAD adimensionales creados en la Sección 1, como `wire_diameter_to_cell_size` y `thick_to_cell_size`.

In [ ]:
# ============================================================
# 2A.6 Gráficos focalizados CAD-física
# ============================================================

CAD_PHYSICS_SCATTER_CONFIG = {
    "BCC": [("D_toro", "surface_area_m2"), ("d_wire", "surface_to_volume_ratio")],
    "EXP": [("square", "surface_area_m2"), ("height", "surface_to_volume_ratio")],
    "KELVIN": [("cell_size", "surface_area_m2"), ("wire_diameter", "surface_to_volume_ratio")],
    "SCHWARZ": [("cell_size", "surface_area_m2"), ("thick", "surface_to_volume_ratio")]
}

for geometry, pairs in CAD_PHYSICS_SCATTER_CONFIG.items():
    df_family_viable = family_dfs[geometry][family_dfs[geometry][BINARY_OUTPUT] == 1].copy()
    for x_col, y_col in pairs:
        if x_col not in df_family_viable.columns or y_col not in df_family_viable.columns:
            continue
        df_plot = df_family_viable.dropna(subset=[x_col, y_col]).copy()
        plt.figure(figsize=(7, 5))
        if sns is not None:
            sns.scatterplot(data=df_plot, x=x_col, y=y_col, alpha=0.75, s=45)
        else:
            plt.scatter(df_plot[x_col], df_plot[y_col], alpha=0.75)
        plt.title(f"{geometry}: {x_col} vs {y_col}")
        plt.xlabel(x_col)
        plt.ylabel(y_col)
        plt.grid(True, alpha=0.3)
        if y_col == "surface_to_volume_ratio" and (df_plot[y_col].dropna() > 0).all():
            plt.yscale("log")
        plt.tight_layout()
        plt.savefig(f"outputs/figures/2A_cad_physics_scatter_{geometry}_{x_col}_{y_col}.png", dpi=300)
        plt.show()

In [ ]:
# ============================================================
# 2A.7 Pairplots opcionales de diagnóstico
# ============================================================

if RUN_PAIRPLOTS:
    # Pairplots físicos solo para EXP y SCHWARZ para no saturar el notebook
    pairplot_config = {
        "EXP": ["square", "height", "surface_area_m2", "relative_density", "surface_to_volume_ratio"],
        "SCHWARZ": ["cell_size", "thick", "surface_area_m2", "relative_density", "surface_to_volume_ratio"]
    }
    for geometry, cols in pairplot_config.items():
        cols = [c for c in cols if c in family_dfs[geometry].columns]
        df_plot = family_dfs[geometry][family_dfs[geometry][BINARY_OUTPUT] == 1][cols].dropna().copy()
        if len(df_plot) > 700:
            df_plot = df_plot.sample(700, random_state=RANDOM_STATE)
        if sns is not None and len(df_plot) > 5:
            g = sns.pairplot(df_plot, vars=cols, corner=True, diag_kind="hist", plot_kws={"alpha": 0.65, "s": 25})
            g.fig.suptitle(f"Relaciones CAD-física en geometrías viables - {geometry}", y=1.02)
            g.fig.savefig(f"outputs/figures/2A_pairplot_physical_{geometry}.png", dpi=300, bbox_inches="tight")
            plt.show()
else:
    print("RUN_PAIRPLOTS=False. Pairplots omitidos por defecto para optimizar ejecución.")

In [ ]:
# ============================================================
# 2A.8 Benchmarking interfamilia
# ============================================================

BENCHMARK_METRICS = ["relative_density", "porosity", "surface_to_volume_ratio", "surface_to_envelope_volume_ratio"]
available_benchmark_metrics = [c for c in BENCHMARK_METRICS if c in df_viable.columns]

benchmark_summary = df_viable.groupby("geometry")[available_benchmark_metrics].agg(["count", "mean", "median", "min", "max", "std"])
display(benchmark_summary)
benchmark_summary.to_excel("outputs/tables/2A_benchmark_summary_by_geometry.xlsx")

for metric in available_benchmark_metrics:
    plt.figure(figsize=(9, 5))
    if sns is not None:
        sns.violinplot(data=df_viable, x="geometry", y=metric, inner="box", cut=0, order=["BCC", "EXP", "KELVIN", "SCHWARZ"])
    else:
        df_viable.boxplot(column=metric, by="geometry", figsize=(9, 5))
        plt.suptitle("")
    plt.title(f"Benchmark interfamilia - {metric}")
    plt.xlabel("Familia geométrica")
    plt.ylabel(metric)
    plt.grid(True, alpha=0.3)
    if metric in ["surface_to_volume_ratio", "surface_to_envelope_volume_ratio"] and (df_viable[metric].dropna() > 0).all():
        plt.yscale("log")
    plt.tight_layout()
    plt.savefig(f"outputs/figures/2A_benchmark_{metric}.png", dpi=300)
    plt.show()

### Benchmarking interfamilia

Las métricas derivadas permiten comparar familias desde una perspectiva de ingeniería: densidad relativa, porosidad, área por volumen sólido y área por volumen envolvente. Estas métricas enriquecen la narrativa de diseño, pero deben manejarse cuidadosamente para evitar fuga de información en modelado.

In [ ]:
# ============================================================
# 2A.9 Pareto absoluto y normalizado
# ============================================================

def pareto_front_min_x_max_y(df, x_col, y_col, eps=1e-12):
    df_p = df.dropna(subset=[x_col, y_col]).copy()
    df_p = df_p[np.isfinite(df_p[x_col]) & np.isfinite(df_p[y_col]) & (df_p[x_col] > 0) & (df_p[y_col] > 0)].copy()
    df_p = df_p.sort_values([x_col, y_col], ascending=[True, False]).reset_index(drop=True)
    best_y = -np.inf
    pareto_indices = []
    for idx, row in df_p.iterrows():
        if row[y_col] > best_y + eps:
            pareto_indices.append(idx)
            best_y = row[y_col]
    df_p["is_pareto"] = False
    df_p.loc[pareto_indices, "is_pareto"] = True
    return df_p


def pareto_geometry_summary(pareto_df):
    summary = pareto_df[pareto_df["is_pareto"]].groupby("geometry").size().reset_index(name="pareto_candidates").sort_values("pareto_candidates", ascending=False)
    total = summary["pareto_candidates"].sum()
    summary["pareto_share_pct"] = 100 * summary["pareto_candidates"] / total if total > 0 else np.nan
    return summary


pareto_abs_df = pareto_front_min_x_max_y(df_viable, "plastic_mass_kg", "surface_area_m2")
pareto_norm_df = pareto_front_min_x_max_y(df_viable, "relative_density", "surface_to_envelope_volume_ratio")

# Pareto absoluto
plt.figure(figsize=(10, 7))
if sns is not None:
    sns.scatterplot(data=pareto_abs_df, x="plastic_mass_kg", y="surface_area_m2", hue="geometry", alpha=0.45, s=45)
    sns.scatterplot(data=pareto_abs_df[pareto_abs_df["is_pareto"]], x="plastic_mass_kg", y="surface_area_m2", color="black", s=90, marker="X", label="Frontera Pareto aproximada")
else:
    plt.scatter(pareto_abs_df["plastic_mass_kg"], pareto_abs_df["surface_area_m2"], alpha=0.45)
plt.title("Pareto absoluto: masa plástica vs área superficial")
plt.xlabel("Masa plástica menor es mejor")
plt.ylabel("Área superficial mayor es mejor")
plt.xscale("log")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("outputs/figures/2A_pareto_absoluto_masa_plastica_area.png", dpi=300)
plt.show()

pareto_abs_geometry_summary = pareto_geometry_summary(pareto_abs_df)
print("Candidatos Pareto absolutos por geometría:")
display(pareto_abs_geometry_summary)

# Pareto normalizado
plt.figure(figsize=(10, 7))
if sns is not None:
    sns.scatterplot(data=pareto_norm_df, x="relative_density", y="surface_to_envelope_volume_ratio", hue="geometry", alpha=0.45, s=45)
    sns.scatterplot(data=pareto_norm_df[pareto_norm_df["is_pareto"]], x="relative_density", y="surface_to_envelope_volume_ratio", color="black", s=90, marker="X", label="Frontera Pareto aproximada")
else:
    plt.scatter(pareto_norm_df["relative_density"], pareto_norm_df["surface_to_envelope_volume_ratio"], alpha=0.45)
plt.title("Pareto normalizado: densidad relativa vs área por volumen envolvente")
plt.xlabel("Densidad relativa menor es mejor")
plt.ylabel("Área por volumen envolvente mayor es mejor")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("outputs/figures/2A_pareto_normalizado_densidad_area_envolvente.png", dpi=300)
plt.show()

pareto_norm_geometry_summary = pareto_geometry_summary(pareto_norm_df)
print("Candidatos Pareto normalizados por geometría:")
display(pareto_norm_geometry_summary)

### Interpretación de Pareto

El Pareto absoluto responde qué diseño entrega mayor área superficial con menor masa plástica. El Pareto normalizado responde qué diseño entrega mayor área por volumen envolvente con menor densidad relativa. Este segundo criterio es más justo para comparar familias, porque reduce el sesgo hacia geometrías grandes.

In [ ]:
# ============================================================
# 2A.10 Diseños destacados con tabla compacta
# ============================================================

def make_cad_params_string(row):
    geometry = row["geometry"]
    cols = INPUT_COLS_BY_GEOMETRY_BASE[geometry] + CAD_DERIVED_INPUTS_BY_GEOMETRY.get(geometry, [])
    parts = []
    for col in cols:
        if col in row.index and pd.notna(row[col]):
            val = row[col]
            parts.append(f"{col}={val:.4g}" if isinstance(val, (float, np.floating)) else f"{col}={val}")
    return "; ".join(parts)


def compact_design_table(df, n=10):
    cols = [
        "geometry", "cad_params", "surface_area_m2", "volume_m3", "envelope_volume_m3",
        "relative_density", "porosity", "surface_to_volume_ratio",
        "surface_to_envelope_volume_ratio", "plastic_mass_kg", "steel_mass_kg", "elapsed_s"
    ]
    out = df.copy()
    out["cad_params"] = out.apply(make_cad_params_string, axis=1)
    return out[[c for c in cols if c in out.columns]].head(n)


top_surface_to_volume = df_viable.dropna(subset=["surface_to_volume_ratio"]).sort_values("surface_to_volume_ratio", ascending=False)
top_porosity = df_viable.dropna(subset=["porosity"]).sort_values("porosity", ascending=False)
top_surface_to_envelope = df_viable.dropna(subset=["surface_to_envelope_volume_ratio"]).sort_values("surface_to_envelope_volume_ratio", ascending=False)
top_pareto_abs = pareto_abs_df[pareto_abs_df["is_pareto"]].sort_values(["plastic_mass_kg", "surface_area_m2"], ascending=[True, False])
top_pareto_norm = pareto_norm_df[pareto_norm_df["is_pareto"]].sort_values(["relative_density", "surface_to_envelope_volume_ratio"], ascending=[True, False])

print("Top diseños por ratio área/volumen:")
display(compact_design_table(top_surface_to_volume, 10))
print("Top diseños por porosidad:")
display(compact_design_table(top_porosity, 10))
print("Candidatos Pareto absolutos:")
display(compact_design_table(top_pareto_abs, 10))
print("Candidatos Pareto normalizados:")
display(compact_design_table(top_pareto_norm, 10))

In [ ]:
# ============================================================
# 2A.11 Redundancia y conclusiones automáticas
# ============================================================

redundancy_cols = ["volume_m3", "plastic_mass_kg", "steel_mass_kg", "surface_area_m2", "relative_density", "surface_to_volume_ratio", "surface_to_envelope_volume_ratio"]
redundancy_cols = [c for c in redundancy_cols if c in df_viable.columns]
redundancy_corr = df_viable[redundancy_cols].corr()

display(redundancy_corr)

plt.figure(figsize=(9, 7))
if sns is not None:
    sns.heatmap(redundancy_corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, square=True)
else:
    plt.imshow(redundancy_corr, cmap="coolwarm", aspect="auto")
    plt.colorbar()
plt.title("Correlación entre volumen, masas y métricas derivadas")
plt.tight_layout()
plt.savefig("outputs/figures/2A_redundancy_volume_mass_corr.png", dpi=300)
plt.show()

eda_conclusions = {}
max_failure_row = dataset_quality_summary.sort_values("failed_pct", ascending=False).iloc[0]
eda_conclusions["geometry_highest_failure_pct"] = max_failure_row["geometry"]
eda_conclusions["highest_failure_pct"] = max_failure_row["failed_pct"]

for metric, geom_key, value_key in [
    ("surface_to_volume_ratio", "best_surface_to_volume_median_geometry", "best_surface_to_volume_median"),
    ("surface_to_envelope_volume_ratio", "best_surface_to_envelope_median_geometry", "best_surface_to_envelope_median"),
    ("porosity", "best_porosity_median_geometry", "best_porosity_median")
]:
    if metric in df_viable.columns:
        s = df_viable.groupby("geometry")[metric].median().sort_values(ascending=False)
        eda_conclusions[geom_key] = s.index[0]
        eda_conclusions[value_key] = s.iloc[0]

eda_conclusions["pareto_absolute_candidate_count"] = int(pareto_abs_df["is_pareto"].sum())
eda_conclusions["pareto_normalized_candidate_count"] = int(pareto_norm_df["is_pareto"].sum())

if len(pareto_abs_geometry_summary) > 0:
    eda_conclusions["pareto_absolute_dominant_geometry"] = pareto_abs_geometry_summary.iloc[0]["geometry"]
    eda_conclusions["pareto_absolute_dominant_share_pct"] = pareto_abs_geometry_summary.iloc[0]["pareto_share_pct"]
if len(pareto_norm_geometry_summary) > 0:
    eda_conclusions["pareto_normalized_dominant_geometry"] = pareto_norm_geometry_summary.iloc[0]["geometry"]
    eda_conclusions["pareto_normalized_dominant_share_pct"] = pareto_norm_geometry_summary.iloc[0]["pareto_share_pct"]

eda_conclusions_df = pd.DataFrame([eda_conclusions])
display(eda_conclusions_df)

## Conclusiones de la Sección 2A

El EDA muestra que las familias geométricas presentan comportamientos diferentes. Schwarz es la familia con mayor porcentaje de fallas, por lo que será prioritaria para clasificación de viabilidad. BCC es muy estable y presenta buen desempeño mediano en eficiencia área/volumen. EXP destaca por su alta porosidad mediana.

Las interacciones físicas confirman que las propiedades dependen de proporciones geométricas, no solo de dimensiones absolutas. Por ello, se agregaron ratios CAD adimensionales por familia.

Los gráficos de Pareto evidencian que no existe una geometría universalmente óptima. La selección depende del criterio: masa, área superficial, porosidad, densidad relativa o eficiencia respecto al volumen envolvente.

La siguiente subsección natural es **2B — Clasificación de viabilidad geométrica**, usando solo variables CAD de entrada, particiones estratificadas y métricas adecuadas para datos desbalanceados.

In [ ]:
# ============================================================
# 2A.12 Exportación final de reportes EDA
# ============================================================

eda_reports_excel_path = "outputs/tables/section2A_eda_reports_iteracion3.xlsx"
highlighted_designs_path = "outputs/tables/2A_highlighted_designs_iteracion3.xlsx"

with pd.ExcelWriter(eda_reports_excel_path, engine="openpyxl") as writer:
    dataset_quality_summary.to_excel(writer, sheet_name="DATASET_QUALITY", index=False)
    modeling_view_report_df.to_excel(writer, sheet_name="MODELING_VIEWS", index=False)
    top_correlations_df.to_excel(writer, sheet_name="TOP_CORRELATIONS", index=False)
    benchmark_summary.to_excel(writer, sheet_name="BENCHMARK_SUMMARY")
    pareto_abs_geometry_summary.to_excel(writer, sheet_name="PARETO_ABS_BY_GEOM", index=False)
    pareto_norm_geometry_summary.to_excel(writer, sheet_name="PARETO_NORM_BY_GEOM", index=False)
    redundancy_corr.to_excel(writer, sheet_name="REDUNDANCY_CORR")
    eda_conclusions_df.to_excel(writer, sheet_name="EDA_CONCLUSIONS", index=False)

with pd.ExcelWriter(highlighted_designs_path, engine="openpyxl") as writer:
    compact_design_table(top_surface_to_volume, 25).to_excel(writer, sheet_name="TOP_SURF_TO_VOLUME", index=False)
    compact_design_table(top_porosity, 25).to_excel(writer, sheet_name="TOP_POROSITY", index=False)
    compact_design_table(top_surface_to_envelope, 25).to_excel(writer, sheet_name="TOP_SURF_TO_ENVELOPE", index=False)
    compact_design_table(top_pareto_abs, 25).to_excel(writer, sheet_name="PARETO_ABSOLUTE", index=False)
    compact_design_table(top_pareto_norm, 25).to_excel(writer, sheet_name="PARETO_NORMALIZED", index=False)

print("Reportes EDA exportados:")
print(eda_reports_excel_path)
print(highlighted_designs_path)

# 2B. Clasificación de viabilidad geométrica

En esta subsección se entrena un conjunto de modelos para predecir la viabilidad geométrica de una combinación CAD.

Para hacer más clara la interpretación, se usará como variable objetivo:

- `failure_bin = 1`: la geometría falla o no es viable.
- `failure_bin = 0`: la geometría reconstruye correctamente.

Esta formulación permite tratar la falla geométrica como la clase positiva del problema. El objetivo práctico es detectar combinaciones paramétricas riesgosas antes de intentar reconstruirlas en SolidWorks.

Decisiones metodológicas:

- Se usan únicamente variables CAD de entrada y ratios CAD adimensionales.
- No se usan variables físicas como `volume_m3`, `surface_area_m2`, masas, tiempos ni métricas derivadas para evitar fuga de información.
- Se evalúa con métricas adecuadas para datos desbalanceados: matriz de confusión, recall de fallas, F1 de fallas, balanced accuracy y PR-AUC.
- BCC se reporta, pero no se toma como caso principal de clasificación porque solo tiene 3 fallas.


In [ ]:
# ============================================================
# 2B.1 Configuración para clasificación de viabilidad
# ============================================================

!pip install -q xgboost

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

# Familias con suficientes fallas para modelado individual.
MIN_FAILURES_FOR_CLASSIFICATION = 20
CLASSIFICATION_FAMILIES = [
    geometry
    for geometry, df_family in family_dfs.items()
    if int((df_family["failure_bin"] == 1).sum()) >= MIN_FAILURES_FOR_CLASSIFICATION
]

print("Familias seleccionadas para clasificación individual:")
print(CLASSIFICATION_FAMILIES)

print("
Familias omitidas como modelo individual principal:")
omitted_families = [g for g in family_dfs.keys() if g not in CLASSIFICATION_FAMILIES]
print(omitted_families)

skf_2b = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


## 2B.2 Variables predictoras permitidas

Para predecir viabilidad geométrica se deben usar solamente variables conocidas antes de ejecutar SolidWorks. Por tanto, las variables predictoras serán:

1. Parámetros CAD originales de cada familia.
2. Ratios CAD adimensionales creados con conocimiento físico, como `wire_diameter_to_cell_size` o `thick_to_cell_size`.

No se usan salidas físicas, métricas derivadas de salidas, masas, tiempos de ejecución ni mensajes de error. Usarlas generaría **data leakage**, porque esas variables solo se conocen después de intentar reconstruir la geometría.


In [ ]:
# ============================================================
# 2B.2 Construcción de vistas para clasificación
# ============================================================

def get_classification_features(geometry, mode="base_plus_ratios"):
    """
    Retorna las columnas de entrada para clasificación.

    mode='base': parámetros CAD originales.
    mode='base_plus_ratios': parámetros CAD originales + ratios CAD adimensionales.
    """
    base_cols = INPUT_COLS_BY_GEOMETRY_BASE[geometry]
    ratio_cols = CAD_DERIVED_INPUTS_BY_GEOMETRY.get(geometry, [])

    if mode == "base":
        return base_cols
    if mode == "base_plus_ratios":
        return base_cols + ratio_cols

    raise ValueError("mode debe ser 'base' o 'base_plus_ratios'.")


def get_failure_classification_view(df_family, geometry, mode="base_plus_ratios"):
    """
    Crea una vista temporal para clasificación de fallas.

    Target:
    - failure_bin = 1: falla geométrica
    - failure_bin = 0: viable
    """
    feature_cols = get_classification_features(geometry, mode=mode)
    available_cols = [col for col in feature_cols if col in df_family.columns]

    required_cols = available_cols + ["failure_bin", "design_group_id"]
    df_view = df_family.dropna(subset=required_cols).copy()

    X = df_view[available_cols].copy()
    y = df_view["failure_bin"].astype(int).copy()
    groups = df_view["design_group_id"].copy()

    return df_view, X, y, groups, available_cols


classification_view_summary = []

for geometry, df_family in family_dfs.items():
    for mode in ["base", "base_plus_ratios"]:
        df_view, X, y, groups, feature_cols = get_failure_classification_view(
            df_family=df_family,
            geometry=geometry,
            mode=mode
        )

        classification_view_summary.append({
            "geometry": geometry,
            "feature_mode": mode,
            "rows": len(df_view),
            "features": len(feature_cols),
            "feature_names": ", ".join(feature_cols),
            "viable_class_0": int((y == 0).sum()),
            "failure_class_1": int((y == 1).sum()),
            "failure_pct": 100 * int((y == 1).sum()) / len(y) if len(y) > 0 else np.nan,
            "unique_design_groups": groups.nunique()
        })

classification_view_summary_df = pd.DataFrame(classification_view_summary)
display(classification_view_summary_df)


## 2B.3 Modelos a comparar

Se comparan cuatro enfoques:

- **DummyClassifier:** modelo ingenuo que sirve como línea base.
- **Logistic Regression balanceada:** modelo interpretable, útil como referencia lineal.
- **Random Forest:** modelo no lineal capaz de capturar interacciones entre parámetros CAD.
- **XGBoost:** modelo de boosting potente para relaciones no lineales y datos tabulares.

La métrica más importante será el desempeño sobre la clase fallida (`failure_bin = 1`), especialmente `recall_failure`, `f1_failure` y `pr_auc`. El accuracy se reporta solo como métrica secundaria.


In [ ]:
# ============================================================
# 2B.3 Funciones de modelos y evaluación
# ============================================================

MODEL_DISPLAY_ORDER = [
    "Dummy majority",
    "Logistic balanced",
    "Random Forest balanced",
    "XGBoost weighted"
]


def build_classifier(model_name, y_train=None):
    """
    Crea un modelo nuevo para cada entrenamiento.
    y_train se usa para calcular scale_pos_weight en XGBoost.
    """
    if model_name == "Dummy majority":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyClassifier(strategy="most_frequent"))
        ])

    if model_name == "Logistic balanced":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ])

    if model_name == "Random Forest balanced":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])

    if model_name == "XGBoost weighted":
        # failure_bin=1 es la clase positiva. scale_pos_weight = negativos / positivos.
        if y_train is None or int((y_train == 1).sum()) == 0:
            scale_pos_weight = 1.0
        else:
            scale_pos_weight = int((y_train == 0).sum()) / int((y_train == 1).sum())

        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.90,
                colsample_bytree=0.90,
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=scale_pos_weight,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])

    raise ValueError(f"Modelo no reconocido: {model_name}")


def get_failure_probability(model, X):
    """Retorna probabilidad de falla, es decir, P(failure_bin=1)."""
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        return proba[:, 1]
    return None


def evaluate_failure_classifier(y_true, y_pred, y_proba_failure=None):
    """
    Evalúa clasificación con failure_bin=1 como clase positiva.

    Matriz de confusión con labels=[0, 1]:
    [[TN, FP],
     [FN, TP]]

    Donde:
    - TN: viable correctamente clasificado
    - FP: viable clasificado como falla, falsa alarma
    - FN: falla clasificada como viable, falla no detectada
    - TP: falla correctamente detectada
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_failure": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_failure": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_failure": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "precision_viable": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "recall_viable": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "f1_viable": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "TN_viable_correct": int(tn),
        "FP_viable_as_failure": int(fp),
        "FN_failure_as_viable": int(fn),
        "TP_failure_correct": int(tp)
    }

    if y_proba_failure is not None and len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, y_proba_failure)
        metrics["pr_auc"] = average_precision_score(y_true, y_proba_failure)
    else:
        metrics["roc_auc"] = np.nan
        metrics["pr_auc"] = np.nan

    return metrics


In [ ]:
# ============================================================
# 2B.4 Entrenamiento y evaluación holdout por familia
# ============================================================

classification_results = []
trained_holdout_models = {}

for geometry in CLASSIFICATION_FAMILIES:
    df_family = family_dfs[geometry]

    for feature_mode in ["base", "base_plus_ratios"]:
        df_view, X, y, groups, feature_cols = get_failure_classification_view(
            df_family=df_family,
            geometry=geometry,
            mode=feature_mode
        )

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.20,
            random_state=RANDOM_STATE,
            stratify=y
        )

        for model_name in MODEL_DISPLAY_ORDER:
            model = build_classifier(model_name, y_train=y_train)
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            y_proba_failure = get_failure_probability(model, X_test)

            metrics = evaluate_failure_classifier(
                y_true=y_test,
                y_pred=y_pred,
                y_proba_failure=y_proba_failure
            )

            result = {
                "geometry": geometry,
                "feature_mode": feature_mode,
                "model": model_name,
                "n_rows": len(df_view),
                "n_features": X.shape[1],
                "features": ", ".join(feature_cols),
                "train_rows": len(X_train),
                "test_rows": len(X_test),
                "failure_train": int((y_train == 1).sum()),
                "failure_test": int((y_test == 1).sum())
            }
            result.update(metrics)
            classification_results.append(result)

            trained_holdout_models[(geometry, feature_mode, model_name)] = {
                "model": model,
                "X_train": X_train,
                "X_test": X_test,
                "y_train": y_train,
                "y_test": y_test,
                "feature_cols": feature_cols
            }

classification_results_df = pd.DataFrame(classification_results)

classification_results_df = classification_results_df.sort_values(
    ["geometry", "f1_failure", "recall_failure", "pr_auc", "balanced_accuracy"],
    ascending=[True, False, False, False, False]
).reset_index(drop=True)

display(classification_results_df)

classification_results_df.to_excel(
    "outputs/tables/2B_classification_holdout_results.xlsx",
    index=False
)


## 2B.5 Selección preliminar de mejores modelos

Para cada familia se selecciona el mejor modelo preliminar priorizando:

1. `f1_failure`: balance entre precision y recall de fallas.
2. `recall_failure`: capacidad de detectar fallas.
3. `pr_auc`: desempeño probabilístico en un problema desbalanceado.
4. `balanced_accuracy`: desempeño promedio entre clases.

Esta selección no reemplaza el criterio de ingeniería. Si detectar fallas es más importante que evitar falsas alarmas, se puede priorizar `recall_failure` sobre `f1_failure`.


In [ ]:
# ============================================================
# 2B.5 Selección preliminar de mejores modelos por familia
# ============================================================

candidate_results = classification_results_df[
    classification_results_df["model"] != "Dummy majority"
].copy()

best_models_df = (
    candidate_results
    .sort_values(
        ["geometry", "f1_failure", "recall_failure", "pr_auc", "balanced_accuracy"],
        ascending=[True, False, False, False, False]
    )
    .groupby("geometry", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(best_models_df[[
    "geometry", "feature_mode", "model", "n_features",
    "accuracy", "balanced_accuracy", "precision_failure", "recall_failure",
    "f1_failure", "pr_auc", "roc_auc", "FN_failure_as_viable", "TP_failure_correct"
]])

best_models_df.to_excel(
    "outputs/tables/2B_best_models_holdout.xlsx",
    index=False
)


In [ ]:
# ============================================================
# 2B.6 Validación cruzada de mejores modelos
# ============================================================

from sklearn.metrics import make_scorer

scoring_2b = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_failure": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_failure": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_failure": make_scorer(f1_score, pos_label=1, zero_division=0),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

cv_rows = []

for _, row in best_models_df.iterrows():
    geometry = row["geometry"]
    feature_mode = row["feature_mode"]
    model_name = row["model"]

    df_view, X, y, groups, feature_cols = get_failure_classification_view(
        df_family=family_dfs[geometry],
        geometry=geometry,
        mode=feature_mode
    )

    model = build_classifier(model_name, y_train=y)

    cv_result = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=skf_2b,
        scoring=scoring_2b,
        n_jobs=-1,
        return_train_score=False
    )

    cv_row = {
        "geometry": geometry,
        "feature_mode": feature_mode,
        "model": model_name,
        "n_rows": len(X),
        "n_features": X.shape[1]
    }

    for metric_name in scoring_2b.keys():
        values = cv_result[f"test_{metric_name}"]
        cv_row[f"{metric_name}_mean"] = np.mean(values)
        cv_row[f"{metric_name}_std"] = np.std(values)

    cv_rows.append(cv_row)

cv_best_models_df = pd.DataFrame(cv_rows)
display(cv_best_models_df)

cv_best_models_df.to_excel(
    "outputs/tables/2B_best_models_cross_validation.xlsx",
    index=False
)


## 2B.7 Matrices de confusión de los mejores modelos

La matriz de confusión se interpreta con `failure_bin = 1` como clase positiva:

- **TN:** diseño viable correctamente clasificado.
- **FP:** diseño viable clasificado como falla. Es una falsa alarma.
- **FN:** falla clasificada como viable. Es el error más crítico, porque dejaría pasar un diseño problemático.
- **TP:** falla correctamente detectada.


In [ ]:
# ============================================================
# 2B.7 Matrices de confusión de mejores modelos
# ============================================================

best_model_objects = {}

for _, row in best_models_df.iterrows():
    geometry = row["geometry"]
    feature_mode = row["feature_mode"]
    model_name = row["model"]

    key = (geometry, feature_mode, model_name)
    info = trained_holdout_models[key]

    model = info["model"]
    X_test = info["X_test"]
    y_test = info["y_test"]
    y_pred = model.predict(X_test)

    best_model_objects[geometry] = info

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Viable", "Falla"]
    )

    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(f"{geometry} - {model_name}
{feature_mode}")
    plt.tight_layout()
    plt.savefig(f"outputs/figures/2B_confusion_matrix_{geometry}.png", dpi=300)
    plt.show()

    print("=" * 80)
    print(f"{geometry} | {model_name} | {feature_mode}")
    print("=" * 80)
    print(classification_report(y_test, y_pred, target_names=["Viable", "Falla"], zero_division=0))


In [ ]:
# ============================================================
# 2B.8 Curvas precision-recall de mejores modelos
# ============================================================

plt.figure(figsize=(8, 6))

for _, row in best_models_df.iterrows():
    geometry = row["geometry"]
    feature_mode = row["feature_mode"]
    model_name = row["model"]

    info = trained_holdout_models[(geometry, feature_mode, model_name)]
    model = info["model"]
    X_test = info["X_test"]
    y_test = info["y_test"]
    y_proba_failure = get_failure_probability(model, X_test)

    if y_proba_failure is None:
        continue

    precision, recall, thresholds = precision_recall_curve(y_test, y_proba_failure)
    pr_auc_value = average_precision_score(y_test, y_proba_failure)

    plt.plot(recall, precision, label=f"{geometry} | AP={pr_auc_value:.3f}")

plt.title("Curvas Precision-Recall para detección de fallas")
plt.xlabel("Recall de fallas")
plt.ylabel("Precision de fallas")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("outputs/figures/2B_precision_recall_best_models.png", dpi=300)
plt.show()


## 2B.9 Threshold tuning

En problemas desbalanceados, el umbral por defecto de 0.50 no siempre es el mejor. Si el objetivo es detectar fallas geométricas, puede ser conveniente reducir el umbral para aumentar el recall de fallas, aceptando más falsas alarmas.

La siguiente celda evalúa umbrales entre 0.05 y 0.95 para los mejores modelos y selecciona el umbral que maximiza `f1_failure`.


In [ ]:
# ============================================================
# 2B.9 Threshold tuning para mejores modelos
# ============================================================

threshold_rows = []
threshold_grid = np.round(np.arange(0.05, 0.96, 0.05), 2)

for _, row in best_models_df.iterrows():
    geometry = row["geometry"]
    feature_mode = row["feature_mode"]
    model_name = row["model"]

    info = trained_holdout_models[(geometry, feature_mode, model_name)]
    model = info["model"]
    X_test = info["X_test"]
    y_test = info["y_test"]
    y_proba_failure = get_failure_probability(model, X_test)

    if y_proba_failure is None:
        continue

    for threshold in threshold_grid:
        y_pred_threshold = (y_proba_failure >= threshold).astype(int)
        metrics = evaluate_failure_classifier(y_test, y_pred_threshold, y_proba_failure)
        threshold_rows.append({
            "geometry": geometry,
            "feature_mode": feature_mode,
            "model": model_name,
            "threshold": threshold,
            "precision_failure": metrics["precision_failure"],
            "recall_failure": metrics["recall_failure"],
            "f1_failure": metrics["f1_failure"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "FP_viable_as_failure": metrics["FP_viable_as_failure"],
            "FN_failure_as_viable": metrics["FN_failure_as_viable"],
            "TP_failure_correct": metrics["TP_failure_correct"]
        })

threshold_tuning_df = pd.DataFrame(threshold_rows)

best_thresholds_df = (
    threshold_tuning_df
    .sort_values(["geometry", "f1_failure", "recall_failure", "balanced_accuracy"], ascending=[True, False, False, False])
    .groupby("geometry", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("Mejores umbrales por familia según F1 de fallas:")
display(best_thresholds_df)

threshold_tuning_df.to_excel("outputs/tables/2B_threshold_tuning_all.xlsx", index=False)
best_thresholds_df.to_excel("outputs/tables/2B_best_thresholds.xlsx", index=False)

priority_geometry = "SCHWARZ" if "SCHWARZ" in threshold_tuning_df["geometry"].unique() else threshold_tuning_df["geometry"].iloc[0]
plot_df = threshold_tuning_df[threshold_tuning_df["geometry"] == priority_geometry]

plt.figure(figsize=(8, 5))
plt.plot(plot_df["threshold"], plot_df["precision_failure"], marker="o", label="Precision falla")
plt.plot(plot_df["threshold"], plot_df["recall_failure"], marker="o", label="Recall falla")
plt.plot(plot_df["threshold"], plot_df["f1_failure"], marker="o", label="F1 falla")
plt.title(f"Threshold tuning - {priority_geometry}")
plt.xlabel("Umbral para clasificar falla")
plt.ylabel("Métrica")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(f"outputs/figures/2B_threshold_tuning_{priority_geometry}.png", dpi=300)
plt.show()


## 2B.10 Importancia de variables

Para interpretar los modelos se calcula la importancia por permutación sobre el conjunto de prueba. Esta técnica mide cuánto cae el desempeño cuando se rompe aleatoriamente la relación entre una variable y la salida.

En esta subsección se usa `average_precision` como métrica de importancia porque la clase fallida es minoritaria y el PR-AUC es más informativo que accuracy en problemas desbalanceados.


In [ ]:
# ============================================================
# 2B.10 Importancia de variables por permutación
# ============================================================

permutation_importance_rows = []

for _, row in best_models_df.iterrows():
    geometry = row["geometry"]
    feature_mode = row["feature_mode"]
    model_name = row["model"]

    info = trained_holdout_models[(geometry, feature_mode, model_name)]
    model = info["model"]
    X_test = info["X_test"]
    y_test = info["y_test"]
    feature_cols = info["feature_cols"]

    perm = permutation_importance(
        estimator=model,
        X=X_test,
        y=y_test,
        scoring="average_precision",
        n_repeats=20,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    importance_df = pd.DataFrame({
        "geometry": geometry,
        "feature_mode": feature_mode,
        "model": model_name,
        "feature": feature_cols,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std
    }).sort_values("importance_mean", ascending=False)

    permutation_importance_rows.append(importance_df)

    plt.figure(figsize=(8, 4.5))
    plot_df = importance_df.sort_values("importance_mean", ascending=True)
    plt.barh(plot_df["feature"], plot_df["importance_mean"])
    plt.title(f"Importancia por permutación - {geometry}
{model_name}")
    plt.xlabel("Caída media en average precision")
    plt.ylabel("Variable CAD")
    plt.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig(f"outputs/figures/2B_permutation_importance_{geometry}.png", dpi=300)
    plt.show()

permutation_importance_df = pd.concat(permutation_importance_rows, ignore_index=True)
display(permutation_importance_df)

permutation_importance_df.to_excel(
    "outputs/tables/2B_permutation_importance.xlsx",
    index=False
)


In [ ]:
# ============================================================
# 2B.11 Conclusiones automáticas de clasificación
# ============================================================

classification_conclusions = []

for _, row in best_models_df.iterrows():
    geometry = row["geometry"]
    cv_row = cv_best_models_df[cv_best_models_df["geometry"] == geometry].iloc[0]
    threshold_row = best_thresholds_df[best_thresholds_df["geometry"] == geometry].iloc[0]

    classification_conclusions.append({
        "geometry": geometry,
        "best_model_holdout": row["model"],
        "feature_mode": row["feature_mode"],
        "holdout_f1_failure": row["f1_failure"],
        "holdout_recall_failure": row["recall_failure"],
        "holdout_pr_auc": row["pr_auc"],
        "cv_f1_failure_mean": cv_row["f1_failure_mean"],
        "cv_recall_failure_mean": cv_row["recall_failure_mean"],
        "cv_pr_auc_mean": cv_row["pr_auc_mean"],
        "best_threshold_f1": threshold_row["threshold"],
        "best_threshold_f1_failure": threshold_row["f1_failure"],
        "best_threshold_recall_failure": threshold_row["recall_failure"]
    })

classification_conclusions_df = pd.DataFrame(classification_conclusions)
display(classification_conclusions_df)

classification_conclusions_df.to_excel(
    "outputs/tables/2B_classification_conclusions.xlsx",
    index=False
)


## Conclusiones de la Sección 2B

La clasificación de viabilidad geométrica se enfocó en las familias EXP, Kelvin y Schwarz, porque BCC tiene muy pocos casos fallidos para entrenar un clasificador individual confiable.

La evaluación se realizó tratando la falla geométrica como clase positiva (`failure_bin = 1`). Esto permite priorizar métricas útiles para el problema: `recall_failure`, `f1_failure` y `pr_auc`. El accuracy se reporta únicamente como métrica secundaria, ya que puede ser engañoso en datasets desbalanceados.

Los modelos lineales sirven como referencia interpretable, pero los modelos no lineales como Random Forest y XGBoost son candidatos más apropiados cuando existen interacciones entre parámetros CAD. La comparación entre variables base y variables base más ratios CAD permite evaluar si los ratios adimensionales aportan información predictiva adicional.

La siguiente subsección será **2C — Regresión de propiedades físicas**, que es el componente más importante del proyecto. Allí se usarán únicamente geometrías viables para predecir propiedades como `volume_m3`, `surface_area_m2` y `envelope_volume_m3`.


In [ ]:
# ============================================================
# 2B.12 Exportación final de reportes de clasificación
# ============================================================

classification_reports_path = "outputs/tables/section2B_classification_reports_iteracion4.xlsx"

with pd.ExcelWriter(classification_reports_path, engine="openpyxl") as writer:
    classification_view_summary_df.to_excel(writer, sheet_name="CLASSIFICATION_VIEWS", index=False)
    classification_results_df.to_excel(writer, sheet_name="HOLDOUT_RESULTS", index=False)
    best_models_df.to_excel(writer, sheet_name="BEST_MODELS_HOLDOUT", index=False)
    cv_best_models_df.to_excel(writer, sheet_name="BEST_MODELS_CV", index=False)
    threshold_tuning_df.to_excel(writer, sheet_name="THRESHOLD_TUNING", index=False)
    best_thresholds_df.to_excel(writer, sheet_name="BEST_THRESHOLDS", index=False)
    permutation_importance_df.to_excel(writer, sheet_name="PERMUTATION_IMPORTANCE", index=False)
    classification_conclusions_df.to_excel(writer, sheet_name="CONCLUSIONS", index=False)

print(f"Reportes de clasificación exportados en: {classification_reports_path}")
